# Lyra 0.1 — Kaggle GPU training on ONE dataset

**Dataset: `Den4ikAI/russian_dialogues_2` (and nothing else).**

The pipeline in this notebook is strictly single-source:

`Den4ikAI/russian_dialogues_2` → cleaning → tokenizer → train/validation/test → training → checkpoints → Russian evaluation.

No OASST, no UltraChat, no Discord, no PersonaChat, no English corpora, no demo samples.
If the dataset cannot be loaded the build fails loudly instead of falling back to another corpus.

## 1. Environment

In [ ]:
from pathlib import Path
import os, sys, json, glob, shutil, subprocess, hashlib, statistics, math

REPO_URL = "https://github.com/er1kosfunpay-sketch/Lyra0.1.git"
PROJECT_DIR = Path("/kaggle/working/Lyra0.1")
RUN_NAME = "ru_full_v1"              # use a new name for a distinct run
AUTO_RESUME = True
ALLOW_DATASET_CHANGE = False         # the corpus is fixed; a change must be deliberate
USE_ATTACHED_DATA = True
RESUME_CHECKPOINT = None             # optionally an exact /kaggle/input/.../latest.pt
DATASET_REPO = "Den4ikAI/russian_dialogues_2"
EPOCHS = 1                           # planned training volume = EPOCHS passes over the train split
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 3e-4
WARMUP_STEPS = 100
SAVE_EVERY = 1000       # latest.pt (resume granularity) every 1000 optimizer steps
ARCHIVE_EVERY = 5000    # rolling checkpoint_step_*.pt archives (multiple of SAVE_EVERY)
EVAL_EVERY = 1000       # validation loss + best.pt
KEEP_LAST_N_CHECKPOINTS = 2
CONFIG = "configs/kaggle.json"
OUTPUT_DIR = PROJECT_DIR / "checkpoints" / RUN_NAME
print({"project": str(PROJECT_DIR), "run": RUN_NAME, "dataset": DATASET_REPO,
       "auto_resume": AUTO_RESUME, "epochs": EPOCHS})

## 2. GPU check

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. In Notebook Settings select Accelerator > GPU, then rerun.")
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB")
props = torch.cuda.get_device_properties(0)
if props.total_memory < 12 * 1024**3:
    raise RuntimeError(f"GPU 0 has {props.total_memory / 1024**3:.1f} GiB; Lyra context=1024 needs a larger GPU profile.")
x = torch.randn(1024, 1024, device="cuda:0"); y = x @ x
print("CUDA matmul:", float(y.mean()), "BF16 supported:", torch.cuda.is_bf16_supported())
del x, y; torch.cuda.empty_cache()

## 3. Clone or update repository

In [ ]:
if PROJECT_DIR.exists() and (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
    raise RuntimeError(f"Refusing to clone over non-empty directory: {PROJECT_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print("Repository:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 4. Install dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[data,dev]"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)

## 5. Check project code

In [ ]:
subprocess.run([sys.executable, "-m", "py_compile",
                "scripts/train.py", "scripts/prepare_data.py", "scripts/train_tokenizer.py",
                "scripts/evaluate_russian.py", "lyra/checkpoint.py", "lyra/data.py",
                "lyra/tokenizer.py", "lyra/generation.py"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, "scripts/colab_healthcheck.py", "--config", CONFIG, "--require-cuda", "--sanity"],
               cwd=PROJECT_DIR, check=True)

## 6. Prepare the dataset — `Den4ikAI/russian_dialogues_2`

The full corpus is used: no `--max-per-source`, no reservoir sampling, no `max_samples`.
Cleaning removes only broken rows (empty/encoding-damaged/meaningless/link-only/non-Russian) and exact duplicates.
An attached Kaggle Dataset / prior notebook output containing `train.jsonl`, `validation.jsonl`, `test.jsonl`
and `dataset_stats.json` is reused **only after verifying it was built from this single source**.

In [ ]:
data_root = PROJECT_DIR / "data/processed"

def load_stats(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8-sig"))
    except (OSError, ValueError):
        return None

def assert_single_source(stats):
    if not stats:
        raise RuntimeError("dataset_stats.json is missing or unreadable; cannot verify the data source.")
    sources = stats.get("source_names") or []
    repo = (stats.get("dataset") or {}).get("repo")
    if sources != ["ru_chat"] or repo != DATASET_REPO:
        raise RuntimeError(
            f"Refusing to train: this data was built from {sources or repo!r}, "
            f"but Lyra must be trained only on {DATASET_REPO}."
        )
    if stats.get("fallback_used"):
        raise RuntimeError("Refusing to train: the dataset build used a fallback source.")
    return stats

attached = []
for train_file in glob.glob("/kaggle/input/**/train.jsonl", recursive=True):
    base = Path(train_file).parent
    val = base / "validation.jsonl"
    if val.is_file():
        attached.append((base, train_file, val))

used_attached = None
if attached and USE_ATTACHED_DATA:
    base, src_train, src_val = attached[0]
    stats = assert_single_source(load_stats(base / "dataset_stats.json"))
    data_root.mkdir(parents=True, exist_ok=True)
    for name in ("train.jsonl", "validation.jsonl", "test.jsonl", "dataset_stats.json"):
        if (base / name).is_file() and (base / name).resolve() != (data_root / name).resolve():
            shutil.copy2(base / name, data_root / name)
    used_attached = str(base)
    print("Using attached dataset (single source verified):", base)

if used_attached is None:
    already = (data_root / "train.jsonl").is_file() and (data_root / "validation.jsonl").is_file()
    if not already:
        # Single source, full corpus. Fails hard if the dataset is unavailable.
        subprocess.run([sys.executable, "scripts/prepare_data.py", "--out", str(data_root)],
                       cwd=PROJECT_DIR, check=True)
    else:
        print("Reusing prepared splits:", data_root)

train_file, val_file = data_root / "train.jsonl", data_root / "validation.jsonl"
test_file = data_root / "test.jsonl"
if not train_file.is_file() or not val_file.is_file():
    raise FileNotFoundError("Required train.jsonl / validation.jsonl missing after dataset preparation.")
dataset_stats = assert_single_source(load_stats(data_root / "dataset_stats.json"))
if dataset_stats.get("build_state") != "SUCCESS":
    raise RuntimeError(f"Dataset build state is {dataset_stats.get('build_state')!r}, expected SUCCESS.")
print(json.dumps({
    "dataset": DATASET_REPO,
    "original_dialogues": dataset_stats.get("original_dialogues"),
    "clean_dialogues": dataset_stats.get("clean_dialogues"),
    "removed_total": dataset_stats.get("removed_total"),
    "rejections_by_reason": dataset_stats.get("rejections_by_reason"),
    "duplicates_removed": dataset_stats.get("duplicates_removed"),
    "splits": {k: v["conversations"] for k, v in dataset_stats.get("splits", {}).items()},
    "used_attached": used_attached,
}, ensure_ascii=False, indent=2))

## 7. Train / verify the tokenizer on that dataset

The vocabulary is learned from Russian conversation text of the same single source,
then verified with real Russian probe sentences (round-trip must be lossless, no `<UNK>`,
every special token a single id). Token counts of the splits drive the training plan.

In [ ]:
tokenizer_path = PROJECT_DIR / "artifacts/tokenizer/tokenizer.json"
tokenizer_stats_path = PROJECT_DIR / "artifacts/tokenizer/tokenizer_stats.json"

from lyra.tokenizer import LyraTokenizer, SPECIAL_TOKENS
from lyra.config import LyraConfig
from lyra.model import LyraModel
cfg = LyraConfig.from_json(CONFIG)

attached_tok = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
if not tokenizer_path.is_file() and attached_tok:
    tokenizer_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(attached_tok[0], tokenizer_path)

need_train = not tokenizer_path.is_file()
if not need_train:
    probe_tok = LyraTokenizer.from_file(tokenizer_path)
    need_train = probe_tok.vocab_size != cfg.vocab_size

if need_train:
    subprocess.run([sys.executable, "scripts/train_tokenizer.py",
                    "--data", str(train_file),
                    "--vocab-size", str(cfg.vocab_size),
                    "--expect-vocab-size", str(cfg.vocab_size),
                    "--out", str(tokenizer_path),
                    "--stats-out", str(tokenizer_stats_path),
                    "--count-splits", "auto"], cwd=PROJECT_DIR, check=True)
elif not tokenizer_stats_path.is_file():
    # Verify an existing/attached tokenizer instead of trusting it blindly.
    subprocess.run([sys.executable, "scripts/train_tokenizer.py",
                    "--data", str(train_file),
                    "--vocab-size", str(cfg.vocab_size),
                    "--expect-vocab-size", str(cfg.vocab_size),
                    "--out", str(tokenizer_path),
                    "--stats-out", str(tokenizer_stats_path),
                    "--count-splits", "auto"], cwd=PROJECT_DIR, check=True)

tok = LyraTokenizer.from_file(tokenizer_path)
if tok.vocab_size != cfg.vocab_size:
    raise RuntimeError(f"Tokenizer vocabulary {tok.vocab_size} != model vocabulary {cfg.vocab_size}.")
for special in SPECIAL_TOKENS:
    tok.id(special)
RUSSIAN_PROBES = ["Привет", "Привет, как дела?", "Здравствуйте! Рад тебя видеть.",
                  "Что ты сейчас делаешь?", "Мне сегодня очень грустно",
                  "Давай немного поговорим", "Я играю в Roblox"]
for probe in RUSSIAN_PROBES:
    ids = tok.encode(probe)
    back = tok.decode(ids)
    if back != probe or "<UNK>" in back:
        raise RuntimeError(f"Tokenizer Russian round-trip failed for {probe!r}: {back!r}")
tokenizer_report = json.loads(tokenizer_stats_path.read_text(encoding="utf-8"))
if not tokenizer_report.get("verification", {}).get("ok", False):
    raise RuntimeError("Tokenizer verification failed; see " + str(tokenizer_stats_path))
tokenizer_fingerprint = hashlib.sha256(tokenizer_path.read_bytes()).hexdigest()
print("Tokenizer OK;", json.dumps({
    "vocab_size": tok.vocab_size, "special_tokens": list(SPECIAL_TOKENS),
    "trained_on": tokenizer_report.get("trained_on_files"),
    "splits": {k: {"tokens": v["tokens"], "messages": v["messages"]}
               for k, v in tokenizer_report.get("splits", {}).items()},
}, ensure_ascii=False))

## 8. Find checkpoint (auto-resume)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_candidates = sorted(glob.glob("/kaggle/input/**/latest.pt", recursive=True),
                               key=lambda p: Path(p).stat().st_mtime, reverse=True)
resume_path = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None
local_latest = OUTPUT_DIR / "latest.pt"
if AUTO_RESUME and resume_path is None:
    if local_latest.is_file():
        resume_path = local_latest
    elif len(checkpoint_candidates) == 1:
        resume_path = Path(checkpoint_candidates[0])
    elif len(checkpoint_candidates) > 1:
        raise RuntimeError("Multiple attached latest.pt checkpoints found; set RESUME_CHECKPOINT to the intended run:\n" + "\n".join(checkpoint_candidates))
print("Resume checkpoint:", str(resume_path) if resume_path else "none; fresh run")
if resume_path:
    prior_log = resume_path.parent / "training.jsonl"
    if prior_log.is_file() and not (OUTPUT_DIR / "training.jsonl").exists():
        shutil.copy2(prior_log, OUTPUT_DIR / "training.jsonl")

## 9. Configure the run and compute the planned training volume

In [ ]:
def inspect_jsonl(path):
    count = messages = chars = 0; langs = {}; turn_counts = []; approx_tokens = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line); count += 1
            ms = row.get("messages", []); messages += len(ms); turn_counts.append(len(ms))
            langs[row.get("lang", "unknown")] = langs.get(row.get("lang", "unknown"), 0) + 1
            for m in ms:
                chars += len(m.get("content", "")); approx_tokens += len(tok.encode(m.get("content", "")))
    return {"examples": count, "messages": messages, "languages": langs,
            "average_turns": round(statistics.mean(turn_counts), 2) if turn_counts else 0,
            "approx_tokens": approx_tokens,
            "average_conversation_chars": round(chars / max(count, 1), 1)}

train_stats, val_stats = inspect_jsonl(train_file), inspect_jsonl(val_file)

model = LyraModel(cfg)
parameter_count = model.parameter_count()
del model

# Planned volume: EPOCHS full passes over the whole train split.
train_tokens = tokenizer_report.get("splits", {}).get("train", {}).get("tokens")
if not train_tokens:
    raise RuntimeError("Train token count missing; rerun the tokenizer cell.")
effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = math.ceil(train_tokens / (effective_batch * cfg.context_length))
TRAIN_STEPS = EPOCHS * steps_per_epoch

print("========================")
print("LYRA TRAINING")
print("========================")
print(json.dumps({
    "dataset": DATASET_REPO,
    "other_datasets": "NONE",
    "model_parameters": parameter_count,
    "tokenizer": str(tokenizer_path),
    "vocabulary": tok.vocab_size,
    "context": cfg.context_length,
    "dataset_dir": str(data_root),
    "original_dialogues": dataset_stats.get("original_dialogues"),
    "clean_dialogues": dataset_stats.get("clean_dialogues"),
    "train": train_stats,
    "validation": val_stats,
    "train_tokens": train_tokens,
    "effective_batch": effective_batch,
    "steps_per_epoch": steps_per_epoch,
    "epochs": EPOCHS,
    "TRAIN_STEPS": TRAIN_STEPS,
    "estimated_token_budget": TRAIN_STEPS * effective_batch * cfg.context_length,
    "gpu": torch.cuda.get_device_name(0),
    "dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16 (GradScaler)",
    "stage": "pretrain",
    "learning_rate": LR,
    "warmup_steps": WARMUP_STEPS,
    "output": str(OUTPUT_DIR),
    "resume": str(resume_path) if resume_path else None,
}, ensure_ascii=False, indent=2))
print("========================")

## 10. Preflight

In [ ]:
assert torch.cuda.is_available() and tokenizer_fingerprint
assert train_stats["examples"] > 0 and val_stats["examples"] > 0
assert TRAIN_STEPS > 0 and EPOCHS >= 1
assert cfg.vocab_size == tok.vocab_size and cfg.parameter_count() == parameter_count
subprocess.run([sys.executable, "scripts/colab_healthcheck.py", "--config", CONFIG,
                "--require-cuda", "--instantiate", "--sanity"], cwd=PROJECT_DIR, check=True)
print("[OK] CUDA / model config / tokenizer / single-source dataset / splits / forward / backward / optimizer")
print(f"READY TO TRAIN: {TRAIN_STEPS} optimizer steps = {EPOCHS} epoch(s) over the full train split")

## 11. Disk budget (must stay under 19 GB)

In [ ]:
import shutil as _shutil
def dir_bytes(p):
    total = 0; p = Path(p)
    if p.exists():
        for f in p.rglob("*"):
            try:
                if f.is_file(): total += f.stat().st_size
            except OSError: pass
    return total
out_pts = sorted(OUTPUT_DIR.glob("*.pt"))
largest_ckpt = max([p.stat().st_size for p in out_pts], default=0)
est_ckpt = largest_ckpt or (parameter_count * 12 + 50_000_000)
planned = KEEP_LAST_N_CHECKPOINTS + 3   # rolling archives + latest.pt + best.pt + final.pt
project_bytes = sum(dir_bytes(PROJECT_DIR / d) for d in ("checkpoints", "data", "artifacts", "logs", "exports"))
projected = project_bytes + max(0, planned - len(out_pts)) * est_ckpt
du = _shutil.disk_usage("/kaggle/working")
budget = {"disk_total_gib": round(du.total / 1024**3, 2), "disk_free_gib": round(du.free / 1024**3, 2),
          "project_now_gib": round(project_bytes / 1024**3, 2), "est_ckpt_gib": round(est_ckpt / 1024**3, 2),
          "planned_files": planned, "projected_total_gib": round(projected / 1024**3, 2), "limit_gib": 19}
print(json.dumps(budget, indent=2))
if projected > 19 * 1024**3:
    raise RuntimeError("Disk budget above 19 GB; lower KEEP_LAST_N_CHECKPOINTS before training.")
print("Disk budget OK")

## 12. Start training (runs the full planned volume)

In [ ]:
cmd = [sys.executable, "scripts/train.py", "--config", CONFIG,
       "--tokenizer", str(tokenizer_path), "--data", str(train_file), "--validation", str(val_file),
       "--stage", "pretrain", "--steps", str(TRAIN_STEPS),
       "--batch-size", str(BATCH_SIZE), "--grad-accum", str(GRAD_ACCUM),
       "--lr", str(LR), "--warmup-steps", str(WARMUP_STEPS),
       "--save-every", str(SAVE_EVERY), "--archive-every", str(ARCHIVE_EVERY),
       "--keep-last-checkpoints", str(KEEP_LAST_N_CHECKPOINTS),
       "--eval-every", str(EVAL_EVERY), "--out", str(OUTPUT_DIR)]
if resume_path:
    cmd += ["--resume", str(resume_path)]
print("Command:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)

## 13. Verify saved checkpoints (latest / best / final)

In [ ]:
from lyra.checkpoint import load_checkpoint

def verify_checkpoint(path, label):
    path = Path(path)
    if not path.is_file() or path.stat().st_size <= 0:
        raise RuntimeError(f"{label} checkpoint missing or empty: {path}")
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if ck.get("config") != cfg.to_dict():
        raise RuntimeError(f"{label}: checkpoint config does not match the Lyra config")
    if ck.get("metadata", {}).get("tokenizer_fingerprint") != tokenizer_fingerprint:
        raise RuntimeError(f"{label}: tokenizer fingerprint mismatch")
    m = LyraModel(cfg); m.load_state_dict(ck["model"], strict=True)
    info = {"label": label, "file": str(path), "bytes": path.stat().st_size, "step": ck.get("step"),
            "epoch": ck.get("epoch"), "stage": ck.get("stage"), "tokens_seen": ck.get("tokens_seen"),
            "best_validation_loss": ck.get("best_validation_loss"), "model_state": "strict load OK"}
    del m, ck
    return info

report = [verify_checkpoint(OUTPUT_DIR / "latest.pt", "latest")]
final_path = OUTPUT_DIR / "final.pt"
if final_path.is_file():
    report.append(verify_checkpoint(final_path, "final"))
else:
    print("WARNING: final.pt not present yet (training did not reach its planned end)")
best_path = OUTPUT_DIR / "best.pt"
if best_path.is_file():
    report.append(verify_checkpoint(best_path, "best"))
print(json.dumps(report, ensure_ascii=False, indent=2))
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 14. Russian conversational evaluation (fresh process, final checkpoint)

Automated report: russian language, decode quality, `<UNK>` count, repetitions,
empty answers, special-token leakage, length, prompt copying and garbage symbols.

In [ ]:
eval_out = PROJECT_DIR / "logs" / "russian_eval.json"
eval_checkpoint = final_path if final_path.is_file() else (OUTPUT_DIR / "latest.pt")
proc = subprocess.run([sys.executable, "scripts/evaluate_russian.py",
                       "--checkpoint", str(eval_checkpoint),
                       "--tokenizer", str(tokenizer_path),
                       "--out", str(eval_out)], cwd=PROJECT_DIR)
evaluation = json.loads(eval_out.read_text(encoding="utf-8")) if eval_out.is_file() else {}
print(json.dumps({"returncode": proc.returncode,
                  "overall": evaluation.get("overall"),
                  "failed_checks": evaluation.get("failed_checks"),
                  "mean_cyrillic_share": evaluation.get("mean_cyrillic_share"),
                  "total_unk_ids": evaluation.get("total_unk_ids"),
                  "empty_answers": evaluation.get("empty_answers"),
                  "special_token_leaks": evaluation.get("special_token_leaks"),
                  "repetition_loops": evaluation.get("repetition_loops"),
                  "report": str(eval_out)}, ensure_ascii=False, indent=2))

## 15. Final run report

In [ ]:
training_log = OUTPUT_DIR / "training.jsonl"
records = []
if training_log.is_file():
    with training_log.open(encoding="utf-8") as fh:
        for line in fh:
            try: records.append(json.loads(line))
            except ValueError: pass
val_records = [r for r in records if "validation_loss" in r]
train_records = [r for r in records if "loss" in r and "validation_loss" not in r]
final_ckpt = torch.load(final_path, map_location="cpu", weights_only=False) if final_path.is_file() else {}
latest_ckpt = torch.load(OUTPUT_DIR / "latest.pt", map_location="cpu", weights_only=False) if (OUTPUT_DIR / "latest.pt").is_file() else {}
summary = {
    "Dataset": DATASET_REPO,
    "Other datasets": "NONE",
    "Original samples": dataset_stats.get("original_dialogues"),
    "Clean samples": dataset_stats.get("clean_dialogues"),
    "Removed samples": dataset_stats.get("removed_total"),
    "Rejections by reason": dataset_stats.get("rejections_by_reason"),
    "Duplicates removed": dataset_stats.get("duplicates_removed"),
    "Train samples": dataset_stats.get("splits", {}).get("train", {}).get("conversations"),
    "Validation samples": dataset_stats.get("splits", {}).get("validation", {}).get("conversations"),
    "Test samples": dataset_stats.get("splits", {}).get("test", {}).get("conversations"),
    "Tokenizer vocabulary": tok.vocab_size,
    "Total tokens": tokenizer_report.get("total_tokens"),
    "Train tokens": train_tokens,
    "Model parameters": parameter_count,
    "Epochs planned": EPOCHS,
    "Optimizer steps planned": TRAIN_STEPS,
    "Optimizer steps done": final_ckpt.get("step", latest_ckpt.get("step")),
    "Tokens seen": final_ckpt.get("tokens_seen", latest_ckpt.get("tokens_seen")),
    "Final train loss": train_records[-1]["loss"] if train_records else None,
    "Final validation loss": val_records[-1]["validation_loss"] if val_records else None,
    "Best validation loss": final_ckpt.get("best_validation_loss"),
    "Final checkpoint": str(final_path) if final_path.is_file() else None,
    "Latest checkpoint": str(OUTPUT_DIR / "latest.pt"),
    "Best checkpoint": str(best_path) if best_path.is_file() else None,
    "Tokenizer": str(tokenizer_path),
    "Russian inference": evaluation.get("overall", "NOT RUN"),
    "Failed checks": evaluation.get("failed_checks"),
}
del final_ckpt, latest_ckpt
run_report_path = OUTPUT_DIR / "run_report.json"
run_report_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Report:", run_report_path)

## 16. Export model

In [ ]:
export_dir = Path("/kaggle/working/lyra_export"); export_dir.mkdir(parents=True, exist_ok=True)
export_source = final_path if final_path.is_file() else (OUTPUT_DIR / "latest.pt")
ckpt = torch.load(export_source, map_location="cpu", weights_only=False)
torch.save(ckpt["model"], export_dir / "model.pt")
(export_dir / "config.json").write_text(json.dumps(cfg.to_dict(), indent=2), encoding="utf-8")
shutil.copy2(tokenizer_path, export_dir / "tokenizer.json")
(export_dir / "generation_config.json").write_text(json.dumps(
    {"preset": "lyra_chat", "temperature": 0.8, "top_p": 0.9, "top_k": 50,
     "repetition_penalty": 1.08, "max_new_tokens": 256, "eos_token": "<END>"}, indent=2), encoding="utf-8")
readme = f'''Lyra 0.1 export (single-dataset run)
Dataset: {DATASET_REPO}
Other datasets: NONE
Model parameters: {parameter_count:,}
Vocabulary size: {cfg.vocab_size}
Context length: {cfg.context_length}
Training stage: {ckpt.get('stage', 'pretrain')}
Training step: {ckpt.get('step')}
Tokens seen: {ckpt.get('tokens_seen')}
Validation loss: {ckpt.get('best_validation_loss')}
Checkpoint source: {export_source}
Weights initialized from scratch; no pretrained model weights used.
'''
(export_dir / "README.txt").write_text(readme, encoding="utf-8")
print("Exported:", [str(p) for p in export_dir.iterdir()])

## 17. Resume in a new Kaggle session

Use **Save Version → Quick Save** so `/kaggle/working` is persisted. To continue, attach the previous
notebook output as an input (Input → Add Input → Notebook Output Files) and rerun the notebook:
with `AUTO_RESUME = True` the newest valid `latest.pt` restores model, optimizer, scheduler, step and RNG state.

A session that already reached `TRAIN_STEPS` prints `Training complete` with the final/best/latest
checkpoint paths and exits without retraining.

Manual resume of a specific file:

```python
RESUME_CHECKPOINT = "/kaggle/input/<attached-output>/Lyra0.1/checkpoints/ru_full_v1/latest.pt"
```